In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Name: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM: 95.0 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "Qwen/Qwen3.5-9B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 16 files: 100%|██████████████████████| 16/16 [00:00<00:00, 682.44it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
import os

from safetensors import safe_open

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00003-of-00004.safetensors
mtp.fc.weight
mtp.layers.0.self_attn.q_proj.weight

Checking model.safetensors-00002-of-00004.safetensors
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight

Checking model.safetensors-00001-of-00004.safetensors

Checking model.safetensors-00004-of-00004.safetensors
mtp.layers.0.input_layernorm.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [8]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

In [9]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
         

In [ ]:
# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1200,                                 # Optimization iterations per block; high accuracy (Production grade)
    group_size=32,                              # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,                                   # Symmetric quantization; enables faster compute and broader engine compatibility
    gradient_accumulate_steps=2,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                        # Processor for multimodal models (vision/audio), if present
    scheme="W4A16",                             # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                               # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=4,                               # Batch size for calibration; keep < 8 to avoid slow execution/high memory overhead
    seqlen=4096,                                # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,                      # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,                  # Compiles forward passes using torch.compile to accelerate calibration (JIT acceleration)
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision (Crucial for VLM accuracy)
    ignore_layers="linear_attn,mtp,visual",     # Excludes linear-attention, MTP, and visual modules from quantization
    layer_config={
        "mtp": {"data_type": "bfloat16"},       # Keeps the MTP module in BF16 precision
        "mtp.fc": {"data_type": "bfloat16"},    # Keeps the MTP fully connected layer in BF16 precision
    },                                          # Preserves BF16 precision for the specified MTP layers
)

2026-10-16 18:35:50 WARNING autoround.py L500: Passing 'gradient_accumulate_steps' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [11]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-10-16 18:35:51 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2
2026-10-16 18:35:51 INFO layer_config_resolver.py L417: Ignored layers: linear_attn, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_a, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_b, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_qkv, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.in_proj_z, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.out_proj, model.visual.blocks.[0-26].attn.proj, model.visual.blocks.[0-26].attn.qkv, model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2, model.visual.merger.linear_fc1, model.visual.merger.linear_fc2, mtp, visual
2

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-10-16 19:54:09 INFO missing_tensors.py L126: Restored 48 tensor(s) from FP16/BF16 to their original FP32 values: model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.A_log, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.norm.weight.
2026-10-16 19:54:10 INFO missing_tensors.py L350: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00,  8.49shard/s]
2026-10-16 19:54:10 INFO missing_te

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-10-16 19:54:36 INFO missing_tensors.py L126: Restored 48 tensor(s) from FP16/BF16 to their original FP32 values: model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.A_log, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.norm.weight.
2026-10-16 19:54:37 INFO missing_tensors.py L350: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00, 425.88shard/s]
2026-10-16 19:54:37 INFO missing_t

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-10-16 19:55:14 INFO missing_tensors.py L126: Restored 48 tensor(s) from FP16/BF16 to their original FP32 values: model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.A_log, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30].linear_attn.norm.weight.
2026-10-16 19:55:15 INFO missing_tensors.py L350: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00, 674.11shard/s]
2026-10-16 19:55:15 INFO missing_t

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1152)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-26): 27 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1152, out_features=3456, bias=True)
             (proj): Linear(in_features=1152, out_features=1152, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
             (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
             (act_fn): 

In [12]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo, wipes existing files, and uploads new folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi(token=token)
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns="*",  # <-- Deletes all existing files in the repo before syncing
            commit_message="Wipe repo and upload new files",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [13]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [14]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"),
        f"{base_name}-W4A16-AutoRound-LLM-Compressor",
        hf_token,
    )

else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w4g32/auto-round-auto-gptq to Vishva007/Qwen3.5-9B-W4A16-AutoRound...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-9B-W4A16-AutoRound

[Hub] Pushing ./AutoRound/local_model-w4g32/auto-gptq to Vishva007/Qwen3.5-9B-W4A16-AutoRound-GPTQ...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-9B-W4A16-AutoRound-GPTQ

[Hub] Pushing ./AutoRound/local_model-w4g32/llm-compressor-wint-a16 to Vishva007/Qwen3.5-9B-W4A16-AutoRound-LLM-Compressor...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-9B-W4A16-AutoRound-LLM-Compressor
